# Approach #2: Fine-tuned BERT

This notebook implements a state-of-the-art transformer approach:
- Fine-tune `bert-base-uncased` from HuggingFace
- Multi-class classification with early stopping
- Rich evaluation metrics and visualizations

## Step 0: Setup for Google Colab (Optional)

Uncomment and run this cell if using Google Colab

In [ ]:
# For Google Colab - uncomment to use
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/My\ Drive/path/to/clinical_notes_classification/code

## Step 1: Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch import nn, optim
import warnings
warnings.filterwarnings('ignore')

from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    f1_score, roc_auc_score, roc_curve, auc, label_binarize
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("\nImports successful!")

## Step 2: Load Data and Prepare Encodings

In [ ]:
# Load datasets
train_data = pd.read_csv('train_data.csv')
val_data = pd.read_csv('validation_data.csv')
test_data = pd.read_csv('test_data.csv')

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Load class mapping
with open('class_mapping.json', 'r') as f:
    class_mapping = json.load(f)

# Create label encoder
le = LabelEncoder()
le.fit(list(class_mapping.keys()))

# Create reverse mapping
reverse_mapping = {v: k for k, v in class_mapping.items()}

print(f"\nClass mapping: {class_mapping}")
print(f"Classes: {le.classes_}")

## Step 3: Tokenization with BERT

In [ ]:
# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_length = 512  # BERT max length

def tokenize_texts(texts, max_length=512):
    """
    Tokenize texts using BERT tokenizer.
    Returns input_ids, attention_masks, and token_type_ids.
    """
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encodings

print("Tokenizing training texts...")
train_encodings = tokenize_texts(train_data['free_text_note'], max_length)

print("Tokenizing validation texts...")
val_encodings = tokenize_texts(val_data['free_text_note'], max_length)

print("Tokenizing test texts...")
test_encodings = tokenize_texts(test_data['free_text_note'], max_length)

# Encode labels
train_labels = torch.tensor(le.transform(train_data['STATUS'].values))
val_labels = torch.tensor(le.transform(val_data['STATUS'].values))
test_labels = torch.tensor(le.transform(test_data['STATUS'].values))

print(f"\n✓ Tokenization complete!")
print(f"  Train: {train_encodings['input_ids'].shape}")
print(f"  Val: {val_encodings['input_ids'].shape}")
print(f"  Test: {test_encodings['input_ids'].shape}")

## Step 4: Create PyTorch Datasets and DataLoaders

In [ ]:
# Create datasets
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    train_labels
)

val_dataset = TensorDataset(
    val_encodings['input_ids'],
    val_encodings['attention_mask'],
    val_labels
)

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    test_labels
)

# Create dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoaders created with batch_size={batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## Step 5: Load BERT Model and Setup Training

In [ ]:
# Load pre-trained BERT for sequence classification
num_labels = len(le.classes_)
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

# Move model to device
model = model.to(device)

# Setup optimizer and scheduler
num_epochs = 5
total_steps = len(train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Loss function
criterion = nn.CrossEntropyLoss()

print(f"Model: BERT Base Uncased")
print(f"Number of labels: {num_labels}")
print(f"Number of epochs: {num_epochs}")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"\nModel loaded and ready for training!")

## Step 6: Training Function with Early Stopping

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, device):
    """
    Train for one epoch.
    """
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = [t.to(device) for t in batch]
        input_ids, attention_mask, labels = batch
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

def evaluate(model, eval_loader, device):
    """
    Evaluate model on validation or test set.
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = [t.to(device) for t in batch]
            input_ids, attention_mask, labels = batch
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            # Get predictions
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(eval_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    return avg_loss, accuracy, weighted_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)

print("Training functions defined!")

## Step 7: Train Model with Early Stopping

In [ ]:
# Early stopping parameters
best_val_f1 = -np.inf
patience = 2
patience_counter = 0

# Store metrics
train_losses = []
val_losses = []
val_f1_scores = []
val_accuracies = []

print("Starting BERT fine-tuning...")
print("="*70)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)
    print(f"  Train Loss: {train_loss:.4f}")
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = evaluate(model, val_loader, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_f1_scores.append(val_f1)
    print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1 (weighted): {val_f1:.4f}")
    
    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_bert_model.pt')
        print(f"  ✓ Best model saved (F1: {val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered")
            break

print("\n" + "="*70)
print("Training complete!")

## Step 8: Training Curves

## Step 9: Load Best Model and Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_bert_model.pt'))
print("✓ Best model loaded")

# Evaluate on all sets
train_loss, train_acc, train_f1, train_preds, train_labels_np, train_probs = evaluate(model, train_loader, device)
val_loss, val_acc, val_f1, val_preds, val_labels_np, val_probs = evaluate(model, val_loader, device)
test_loss, test_acc, test_f1, test_preds, test_labels_np, test_probs = evaluate(model, test_loader, device)

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"\nTrain Set:")
print(f"  Accuracy: {train_acc:.4f}, Weighted F1: {train_f1:.4f}")
print(f"\nValidation Set:")
print(f"  Accuracy: {val_acc:.4f}, Weighted F1: {val_f1:.4f}")
print(f"\nTest Set:")
print(f"  Accuracy: {test_acc:.4f}, Weighted F1: {test_f1:.4f}")

## Step 10: Detailed Evaluation Metrics

In [ ]:
# Convert predictions to class labels
test_preds_labels = le.inverse_transform(test_preds)
test_labels_labels = le.inverse_transform(test_labels_np)

# Calculate macro F1
test_macro_f1 = f1_score(test_labels_np, test_preds, average='macro', zero_division=0)

print("\nTest Set - Detailed Classification Report:")
print(classification_report(test_labels_labels, test_preds_labels, zero_division=0))

# Confusion matrix
test_cm = confusion_matrix(test_labels_labels, test_preds_labels, labels=le.classes_)

print(f"\nTest Set - Additional Metrics:")
print(f"  Weighted F1: {test_f1:.4f}")
print(f"  Macro F1: {test_macro_f1:.4f}")
print(f"  Accuracy: {test_acc:.4f}")

## Step 11: Confusion Matrices

In [ ]:
# Get confusion matrices for all sets
train_preds_labels = le.inverse_transform(train_preds)
train_labels_labels = le.inverse_transform(train_labels_np)
train_cm = confusion_matrix(train_labels_labels, train_preds_labels, labels=le.classes_)

val_preds_labels = le.inverse_transform(val_preds)
val_labels_labels = le.inverse_transform(val_labels_np)
val_cm = confusion_matrix(val_labels_labels, val_preds_labels, labels=le.classes_)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, cm, title in [
    (axes[0], train_cm, 'Train Set'),
    (axes[1], val_cm, 'Validation Set'),
    (axes[2], test_cm, 'Test Set')
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_, cbar=False)
    ax.set_title(f'{title} Confusion Matrix', fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('bert_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Confusion matrices saved as 'bert_confusion_matrices.png'")

## Step 12: ROC-AUC Curves

## Step 13: Performance Summary

In [ ]:
# Create summary table
train_macro_f1 = f1_score(train_labels_np, train_preds, average='macro', zero_division=0)
val_macro_f1 = f1_score(val_labels_np, val_preds, average='macro', zero_division=0)

summary_data = {
    'Set': ['Train', 'Validation', 'Test'],
    'Accuracy': [train_acc, val_acc, test_acc],
    'Weighted F1': [train_f1, val_f1, test_f1],
    'Macro F1': [train_macro_f1, val_macro_f1, test_macro_f1]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*70)
print("BERT - PERFORMANCE SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_df))
width = 0.25

ax.bar(x - width, summary_df['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x, summary_df['Weighted F1'], width, label='Weighted F1', alpha=0.8)
ax.bar(x + width, summary_df['Macro F1'], width, label='Macro F1', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('BERT - Performance Metrics', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Set'])
ax.legend()
ax.set_ylim([0, 1])
ax.grid(alpha=0.3, axis='y')

for i, (acc, wf1, mf1) in enumerate(zip(summary_df['Accuracy'], summary_df['Weighted F1'], summary_df['Macro F1'])):
    ax.text(i - width, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=8)
    ax.text(i, wf1 + 0.02, f'{wf1:.3f}', ha='center', fontsize=8)
    ax.text(i + width, mf1 + 0.02, f'{mf1:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('bert_performance_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Performance summary saved as 'bert_performance_summary.png'")

## Step 14: Save Model and Metrics

In [ ]:
# Save model
model.save_pretrained('bert_model_finetuned')
tokenizer.save_pretrained('bert_model_finetuned')
print("✓ Fine-tuned BERT model and tokenizer saved to 'bert_model_finetuned/'")

# Save metrics
metrics = {
    'model_type': 'BERT Fine-tuned',
    'test_accuracy': float(test_acc),
    'test_weighted_f1': float(test_f1),
    'test_macro_f1': float(test_macro_f1),
    'val_accuracy': float(val_acc),
    'val_weighted_f1': float(val_f1),
    'train_accuracy': float(train_acc),
    'train_weighted_f1': float(train_f1)
}

with open('bert_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ Metrics saved to 'bert_metrics.json'")

print("\n" + "="*70)
print("MODEL TRAINING AND EVALUATION COMPLETE")
print("="*70)
print(f"\nKey Results (Test Set):")
print(f"  Weighted F1: {test_f1:.4f}")
print(f"  Macro F1   : {test_macro_f1:.4f}")
print(f"  Accuracy   : {test_acc:.4f}")
print("\nReady for model comparison!")
print("="*70)